# NB1 · The toric code as a decoding problem (Lesson 1)

**Before running this notebook**, write the five `TODO` functions in `msd/toric.py`:

| # | Function | Notes section |
|---|---|---|
| 1 | `toric_checks(d)` → `(H_Z, H_X)` | The lattice T_d; chains |
| 2 | `toric_logicals(d)` → `(L_X, L_Z)` | Chains (logicals) |
| 3 | `coset_test(e, e_hat, H_Z, L_Z)` | Decoding as a coset problem |
| 4 | `min_weight_table(H_Z, w_max)` | Coset problem; tie-break rule in the docstring |
| 5 | `decoding_radius(decode, H_Z, L_Z, w_max, keep=None)` | ρ and ρ_nd |

Work test by test: `pytest tests/test_toric.py -q -k checks`, then `-k logicals`, `-k coset`, `-k table`, `-k radius`.
When the whole file is green, run this notebook top to bottom. Each part states the output you should see.

Help ladder: work alone, then ask for a hint (nudge, key idea, outline), then the advisor. Solutions only on request.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, pathlib, itertools
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "msd").exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from msd.toric import *

## A. Build T_d (TODO 1) · supports E1

Expected: both matrices are d² × 2d², both have rank d² − 1, and (n, k) = (2d², 2).

In [ ]:
for d in (3, 4, 5):
    H_Z, H_X = toric_checks(d)
    css_ok = not (H_X.astype(int) @ H_Z.T.astype(int) % 2).any()
    print(f"T_{d}: H_Z {H_Z.shape}, H_X {H_X.shape}, ranks {gf2_rank(H_Z)}, {gf2_rank(H_X)}, "
          f"H_X H_Z^T = 0: {css_ok}, (n, k) = {css_parameters(H_X, H_Z)}")

The plaquette error in Plan A's labels: e₁ = h(0,0), e₂ = u(1,0) and its twin e₃ = h(0,1), e₄ = u(0,0).
Expected: the syndrome is {v₀, v₂} = {v(0,0), v(1,1)}, the same for both, and e + e′ is the boundary of f(0,0).

In [ ]:
d = 5
H_Z, H_X = toric_checks(d)
n = 2 * d * d
e1, e2 = hedge_index(d, 0, 0), vedge_index(d, 1, 0)
e3, e4 = hedge_index(d, 0, 1), vedge_index(d, 0, 0)
e = support_to_vector(n, (e1, e2))
twin = support_to_vector(n, (e3, e4))
s = syndrome_of(H_Z, e)

print("syndrome of e     :", vector_to_support(s), " expected", (vertex_index(d, 0, 0), vertex_index(d, 1, 1)))
print("syndrome of twin  :", vector_to_support(syndrome_of(H_Z, twin)))
print("e + twin = ∂f(0,0):", vector_to_support(e ^ twin) == vector_to_support(H_X[face_index(d, 0, 0)]))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_lattice(d, error=(e1, e2), syndrome=vector_to_support(s), labels=True, ax=axes[0],
             title="plaquette error e1 + e2 (red), syndrome (squares)")
plot_lattice(d, error=(e1, e2), correction=(e3, e4), syndrome=vector_to_support(s), ax=axes[1],
             title="twin e3 + e4 (blue dashed): same syndrome")
plt.tight_layout(); plt.show()

## B. Logicals (TODO 2) and the coset test (TODO 3) · supports E2

Green: the loops L_X (left) and the cuts L_Z (right). Each loop crosses its own cut once.

In [ ]:
L_X, L_Z = toric_logicals(d)
print("L_X L_Z^T =\n", L_X.astype(int) @ L_Z.T.astype(int) % 2)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_lattice(d, highlight=vector_to_support(L_X[0] | L_X[1]), ax=axes[0], title="L_X: loops h(·,0) and u(0,·)")
plot_lattice(d, highlight=vector_to_support(L_Z[0] | L_Z[1]), ax=axes[1], title="L_Z: cuts h(0,·) and u(·,0)")
plt.tight_layout(); plt.show()

Expected outcomes, in order: success, success, logical, invalid, invalid.

In [ ]:
zero = np.zeros(n, dtype=np.uint8)
cases = {
    "e, corrected by itself":                 (e, e),
    "e, corrected by its twin":               (e, twin),
    "e, corrected by twin + horizontal loop": (e, twin ^ L_X[0]),
    "e, corrected by e1 only":                (e, support_to_vector(n, (e1,))),
    "e, decoder gives up (None)":             (e, None),
}
for name, (err, corr) in cases.items():
    print(f"{name:42s} -> {coset_test(err, corr, H_Z, L_Z)}")

## C. Minimum-weight decoding (TODO 4)

Expected: 17,151 distinct syndromes on T₅ up to weight 3, and the plaquette syndrome decodes to e itself,
(0, 26), because it comes first in `itertools.combinations` order.

In [ ]:
table5 = min_weight_table(H_Z, 3)
decode5 = make_table_decoder(table5, n)
print("distinct syndromes up to weight 3 on T_5:", len(table5))
e_hat = decode5(s)
print("correction for the plaquette syndrome:", vector_to_support(e_hat), "->", coset_test(e, e_hat, H_Z, L_Z))

## D. Radius of minimum-weight decoding (TODO 5) · E2

Expected ρ = ⌊(d−1)/2⌋: 1, 1, 2 on T₃, T₄, T₅. Each witness has weight ρ + 1 and fails as a logical:
the decoder's lighter (or equally light) correction closes a loop around the torus.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, dd in zip(axes, (3, 4, 5)):
    HZ, _ = toric_checks(dd)
    _, LZ = toric_logicals(dd)
    nn = 2 * dd * dd
    t = (dd - 1) // 2
    dec = make_table_decoder(min_weight_table(HZ, t + 1), nn)
    rho, witness = decoding_radius(dec, HZ, LZ, t + 1)
    w_vec = support_to_vector(nn, witness)
    corr = dec(syndrome_of(HZ, w_vec))
    outcome = coset_test(w_vec, corr, HZ, LZ)
    print(f"T_{dd}: rho = {rho} (floor((d-1)/2) = {t}), witness {witness}, outcome {outcome}")
    plot_lattice(dd, error=witness, correction=vector_to_support(corr),
                 syndrome=vector_to_support(syndrome_of(HZ, w_vec)), ax=ax,
                 title=f"T_{dd}: witness (red), correction (blue)")
plt.tight_layout(); plt.show()

## E. Degenerate weight-2 errors · check E5 **after** doing it by hand

Degenerate in the sense of du Crest's Definition 2 (no coset condition): some other error of weight ≤ 2
has the same syndrome. The cell also reports how many of them have every such twin in the *same* coset.
Compare both columns with your answer to E5, including d = 3 and d = 4.

In [ ]:
def degenerate_weight2(dd):
    HZ, _ = toric_checks(dd)
    _, LZ = toric_logicals(dd)
    nn = 2 * dd * dd
    by_syndrome = {}
    for w in range(3):
        for sup in itertools.combinations(range(nn), w):
            key = syndrome_key(syndrome_of(HZ, support_to_vector(nn, sup)))
            by_syndrome.setdefault(key, []).append(sup)
    degenerate, same_coset = [], 0
    for sup in itertools.combinations(range(nn), 2):
        ev = support_to_vector(nn, sup)
        others = [o for o in by_syndrome[syndrome_key(syndrome_of(HZ, ev))] if o != sup]
        if others:
            degenerate.append(sup)
            same_coset += all(coset_test(ev, support_to_vector(nn, o), HZ, LZ) == "success" for o in others)
    return degenerate, same_coset

print(" d   degenerate   twins all in the same coset")
for dd in (3, 4, 5, 6, 7):
    deg, same = degenerate_weight2(dd)
    print(f"{dd:2d}   {len(deg):10d}   {same:10d}")

Spoiler for E5(a): every degenerate weight-2 error on T₅ that contains the edge h(0,0).

In [ ]:
deg5, _ = degenerate_weight2(5)
through0 = [sup for sup in deg5 if 0 in sup]
fig, axes = plt.subplots(1, len(through0), figsize=(3.2 * len(through0), 3.4))
for ax, sup in zip(np.atleast_1d(axes), through0):
    ev = support_to_vector(50, sup)
    plot_lattice(5, error=sup, syndrome=vector_to_support(syndrome_of(H_Z, ev)), ax=ax, title=str(sup))
plt.tight_layout(); plt.show()

## F. Hypergraph products · E3

Expected: HGP(R₃, R₃) = [[13, 1]] and HGP(C_d, C_d) = [[2d², 2]], with row weights at most 4.
The pictures show the two check matrices of HGP(R₃, R₃); columns 0–8 are the bit × bit block, 9–12 the check × check block.

In [ ]:
R3 = repetition_code(3)
HX13, HZ13 = hypergraph_product(R3, R3)
print("HGP(R3, R3): (n, k) =", css_parameters(HX13, HZ13),
      " row weights H_X", sorted(set(int(w) for w in HX13.sum(1))), " H_Z", sorted(set(int(w) for w in HZ13.sum(1))))
for dd in (3, 4, 5):
    HXc, HZc = hypergraph_product(cycle_code(dd), cycle_code(dd))
    print(f"HGP(C_{dd}, C_{dd}): (n, k) = {css_parameters(HXc, HZc)}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, M, name in zip(axes, (HX13, HZ13), ("H_X", "H_Z")):
    ax.imshow(M, cmap="Greys", aspect="auto")
    ax.set_title(f"{name} of HGP(R3, R3)")
    ax.set_xticks(range(13)); ax.set_yticks(range(M.shape[0]))
plt.tight_layout(); plt.show()

## G. The low-noise expansion · E4

F_w = number of weight-w errors on which minimum-weight decoding fails, on T₅ up to w = 4.
Expected: F₀ = F₁ = F₂ = 0 and F₃ > 0, so the slope is ρ + 1 = 3. This cell takes seconds to a minute,
depending on how fast your `min_weight_table` is.
Look at where the F₄ term stops mattering: that is how small p must be before the slope you read off a plot is the true one.

In [ ]:
table5_4 = min_weight_table(H_Z, 4)
F = failing_counts(make_table_decoder(table5_4, n), H_Z, L_Z, 4)
print("F_w on T_5, w = 0..4:", F)

ps = np.logspace(-4, -0.7, 60)
terms = np.array([logical_rate_terms(F, n, p) for p in ps])
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(ps, terms[:, 3], label="F_3 term")
ax.loglog(ps, terms[:, 4], label="F_4 term")
ax.loglog(ps, terms[:, 3] + terms[:, 4], "k--", lw=1, label="sum up to w = 4")
ax.set_xlabel("p"); ax.set_ylabel("contribution to P_L"); ax.legend(); ax.set_title("T_5, minimum-weight decoding")
plt.show()
for p in (1e-1, 1e-2, 1e-3):
    t = logical_rate_terms(F, n, p)
    print(f"p = {p:g}: F_4 term / F_3 term = {t[4] / t[3]:.3f}")